# E006 training on Kaggle -- fold 0 (hard-negative mining)

CODING_SPEC_04_MODEL_SIDE.md: identical recipe to E004-f0 except the
crop sampler mix (45% positive / 30% hard-negative / 25% random, see
configs/E006-f0.yaml) -- the single variable this experiment isolates.

Setup (manual, one time):
1. Attach the `potcharapanchumrak/solar-k-package` dataset (make sure
   `data/interim/hard_negatives_v1.csv` is included -- rebuild+repush with
   `scripts/package_kaggle.py` after `scripts/build_hard_negatives.py` has
   run on all 5 folds' OOF) and the `filament-segmentation-2026` competition.
2. Turn on a GPU accelerator and Internet access.

Checkpoints/logs/probs write to `/kaggle/working/outputs`.

In [ ]:
import os
import time

PKG_DIR = '/kaggle/input/datasets/potcharapanchumrak/solar-k-package'

for attempt in range(10):
    if os.path.isdir(PKG_DIR):
        break
    print(f'{PKG_DIR} not mounted yet, waiting (attempt {attempt + 1}/10)...')
    time.sleep(3)
else:
    raise FileNotFoundError(f'{PKG_DIR} never appeared after 30s -- check the dataset is attached')

os.chdir(PKG_DIR)
print(os.listdir('.'))

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os

COMP_ROOT = '/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026'
PKG_SPLITS = os.path.join(PKG_DIR, 'data', 'splits')
DATA_LINK_ROOT = '/kaggle/working/data_root'
os.makedirs(DATA_LINK_ROOT, exist_ok=True)
for name, target in [('raw', COMP_ROOT), ('splits', PKG_SPLITS)]:
    link = os.path.join(DATA_LINK_ROOT, name)
    if not os.path.islink(link):
        os.symlink(target, link)

os.environ['SOLAR_K_DATA'] = DATA_LINK_ROOT
os.environ['SOLAR_K_OUTPUT'] = '/kaggle/working/outputs'
os.environ['SOLAR_K_CACHE'] = '/kaggle/temp/solar_k_cache'

RESUME_FROM = None
MAX_HOURS = 8.5

In [ ]:
import subprocess

cmd = ['python', 'scripts/train.py', '--config', 'configs/E006-f0.yaml', '--max_hours', str(MAX_HOURS)]
if RESUME_FROM:
    cmd += ['--resume', RESUME_FROM]

# data/interim/hard_negatives_v1.csv must be in the package dataset (it's
# under data/interim/, not included in scripts/package_kaggle.py's default
# INCLUDE list -- add it there, or this run falls back to no hard negatives
# since train.py prints a warning and continues with 0 catalogued stems
# rather than failing).
subprocess.run(cmd, check=True)